# DLQ ( Dead Letter Queue )

## Error Logs & Error tables

DLQ is the process to split bad data from good data, but a mature process can also save logs to answer why is happening the error, and use error tables to check how often they occur.

### import libraries

In [25]:
from IPython.display        import display, HTML

#import findspark
#findspark.init()
import pyspark

#from pyspark.context import SparkContext
from pyspark.sql            import SparkSession
from pyspark.sql.functions  import col, concat, explode, expr, lit, udf, when 
from pyspark.sql.types      import Row, StringType, LongType, ArrayType

In [17]:
import pyspark.sql.functions as F

In [3]:
pyspark.__version__

'4.2.0'

In [4]:
import sys
print( sys.version )

print( sys.version_info )

3.14.4 (main, Jun 18 2026, 14:25:02) [GCC 15.2.0]
sys.version_info(major=3, minor=14, micro=4, releaselevel='final', serial=0)


### create spark session

In [5]:
from pyspark.sql import SparkSession

# Initialize Spark with Delta Lake support
spark = SparkSession.builder \
    .appName("SchemaEvolutionExample") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/25 18:07:57 WARN Utils: Your hostname, heartwings, resolves to a loopback address: 127.0.1.1; using 192.168.1.10 instead (on interface wlo1)
26/08/25 18:07:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/art/git/data_engineer/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/art/.ivy2.5.2/cache
The jars for the packages stored in: /home/art/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-28179e95-542a-42e7-91ce-99e2bf258607;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.log

### read raw data, good and bad

In [14]:
source_format = 'json'
source_path = '/home/art/git/data_engineer/spark/animal_3_raw.json'
df_raw = ( spark.read
    .format(source_format)
    .option("multiLine", "true")
    .load( source_path )
          )

In [26]:
df_raw.show( 10, False )

+---+------+--------+
|age|name  |specie  |
+---+------+--------+
|5  |Clowy |cat     |
|4  |Drako |dog     |
|-2 |Dumbo |elephant|
|-3 |Bullet|falcon  |
+---+------+--------+



### bad data has age as negative number

In [31]:
df = df_raw.withColumn( 'validation_errors', lit( '' ) )
df.show( 10, False )

+---+------+--------+-----------------+
|age|name  |specie  |validation_errors|
+---+------+--------+-----------------+
|5  |Clowy |cat     |                 |
|4  |Drako |dog     |                 |
|-2 |Dumbo |elephant|                 |
|-3 |Bullet|falcon  |                 |
+---+------+--------+-----------------+



### detect bad rows with wrong age

In [33]:
df = df.withColumn( 'validation_errors', 
  when( expr("age is not null and age < 0")  , 
        concat( col( "validation_errors" ), lit("AGE_IS_INVALID;"))
  ) # Changed to use concat function

  .otherwise(col("validation_errors"))             
             )

In [34]:
df.show( 10, False )

+---+------+--------+-----------------+
|age|name  |specie  |validation_errors|
+---+------+--------+-----------------+
|5  |Clowy |cat     |                 |
|4  |Drako |dog     |                 |
|-2 |Dumbo |elephant|AGE_IS_INVALID;  |
|-3 |Bullet|falcon  |AGE_IS_INVALID;  |
+---+------+--------+-----------------+



### here flag valid and invalid rows

In [39]:
df = df.withColumn( 'is_valid',
                   
  when( expr( 'validation_errors is not null and length(validation_errors) > 0 ' ), 'False' )

  .otherwise( lit( 'True' ) )
    
)

In [40]:
df.show( 10, False )

+---+------+--------+-----------------+--------+
|age|name  |specie  |validation_errors|is_valid|
+---+------+--------+-----------------+--------+
|5  |Clowy |cat     |                 |True    |
|4  |Drako |dog     |                 |True    |
|-2 |Dumbo |elephant|AGE_IS_INVALID;  |False   |
|-3 |Bullet|falcon  |AGE_IS_INVALID;  |False   |
+---+------+--------+-----------------+--------+

